# ch06 — Gen5: MambaTSAD faithful vs fixed

Quantify the effect of the four implementation issues (state indexing, CPU branch, HP-filter objective, global-FFT AMA).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
from tsad_forge.models.registry import get_model
from tsad_forge.evaluation.metrics import compute_metrics
from tsad_forge.evaluation.protocol import zscore_normalize

ds = generate_synthetic(n_events=6, seed=6)
train, test = zscore_normalize(ds.train, ds.test)
for name in ["mamba_tsad_faithful", "mamba_tsad_fixed"]:
    vals = []
    for seed in range(3):
        s = get_model(name, seed=seed, epochs=3, window=32).fit(train).score(test)
        vals.append(compute_metrics(s, ds.labels)["vus_pr"])
    print(f"{name:22s} VUS-PR = {np.mean(vals):.3f} +/- {np.std(vals):.3f} (3 seeds)")

In [ ]:
# HP-filter decomposition — faithful feeds the trend, fixed feeds the cycle
from tsad_forge.models.gen5_ssm_foundation.mamba_tsad import hp_filter
trend, cycle = hp_filter(ds.test)
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
for ax, (y, ttl) in zip(axes, [(ds.test[:, 0], "original"),
                               (trend[:, 0], "trend (faithful input)"),
                               (cycle[:, 0], "cycle (fixed input)")]):
    ax.plot(y, lw=0.6)
    ax.set_title(ttl)
plt.tight_layout()